# Netflix Customer Churn Prediction

---

| Field | Details |
|---|---|
| **Student Name** | Telukuntla Yashwanth |
| **Program** | IBM SkillsBuild Data Analytics with AI — Academic Internship |
| **Submitted Under** | BharatCares & AICTE |
| **Dataset** | `netflix_customer_churn.csv` |
| **Notebook File** | `TELUKUNTLA-YASHWANTH_NetflixCustomerChurn.ipynb` |
| **Language** | Python 3 |
| **Random State** | 42 |

---
## Problem Statement

Customer churn — when a subscriber cancels or stops using a service — is one of the most critical business problems for subscription-based platforms like Netflix. Acquiring a new customer costs significantly more than retaining an existing one. This project aims to **identify which customers are at risk of churning** using historical behavioral and account data, and to **build machine learning models** that can predict churn with high accuracy.

---
## Objectives

1. Explore and understand the Netflix customer dataset.
2. Identify key factors that influence customer churn.
3. Perform thorough exploratory data analysis (EDA) with visualisations.
4. Preprocess data (encoding, scaling, splitting) for machine learning.
5. Train three classification models: Logistic Regression, Decision Tree, and Random Forest.
6. Evaluate and compare models using Accuracy, Precision, Recall, F1-Score, ROC-AUC, and Confusion Matrix.
7. Identify the best-performing model and its most important features.
8. Derive actionable business insights to reduce churn.

---
## Dataset Description

The dataset `netflix_customer_churn.csv` contains **5,000 records** of Netflix customers with **14 columns**:

| Column | Type | Description |
|---|---|---|
| `customer_id` | string | Unique customer identifier (dropped before ML) |
| `age` | int | Customer age (18–70) |
| `gender` | string | Male / Female / Other |
| `subscription_type` | string | Basic / Standard / Premium |
| `watch_hours` | float | Total hours watched on the platform |
| `last_login_days` | int | Number of days since last login |
| `region` | string | Geographic region (6 regions) |
| `device` | string | Primary device used (TV / Mobile / Laptop / Desktop / Tablet) |
| `monthly_fee` | float | Monthly subscription fee in USD |
| `payment_method` | string | Payment method used |
| `number_of_profiles` | int | Number of profiles on the account (1–5) |
| `avg_watch_time_per_day` | float | Average daily watch time in hours |
| `favorite_genre` | string | Customer's favourite content genre |
| **`churned`** | int | **Target: 0 = Active, 1 = Churned** |

---
## Section 1 — Import Libraries

In [ ]:
# ── Standard libraries ─────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

# ── Visualisation ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Preprocessing ──────────────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# ── Models ─────────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

# ── Evaluation metrics ─────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

# ── Global settings ────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

print('All libraries imported successfully.')

---
## Section 2 — Load Data

In [ ]:
# Load the dataset
df = pd.read_csv('netflix_customer_churn.csv')

print(f'Dataset loaded successfully.')
print(f'Shape : {df.shape[0]} rows  x  {df.shape[1]} columns')

---
## Section 3 — Inspect Data

In [ ]:
# First five rows
print('First 5 rows:')
df.head()

In [ ]:
# Column names and data types
print('Column Names and Data Types')
print('=' * 40)
print(df.dtypes)

In [ ]:
# Detailed info
df.info()

In [ ]:
# Statistical summary — numerical columns
print('Numerical Features — Descriptive Statistics')
df.describe().T.style.background_gradient(cmap='Blues')

In [ ]:
# Categorical feature unique values
cat_cols_inspect = [c for c in df.select_dtypes(include='object').columns if c != 'customer_id']
for col in cat_cols_inspect:
    print(f'{col} ({df[col].nunique()} unique): {df[col].unique().tolist()}')

---
## Section 4 — Data Cleaning

In [ ]:
# ── Auto-detect target column ──────────────────────────────────────────────
churn_candidates = [c for c in df.columns
                    if any(kw in c.lower() for kw in ['churn', 'cancel', 'status', 'active', 'leave'])]
TARGET = churn_candidates[0] if churn_candidates else None
print(f'Auto-detected target column : "{TARGET}"')
print(f'Unique values               : {df[TARGET].unique().tolist()}')
print(f'Value counts:\n{df[TARGET].value_counts()}')

In [ ]:
# ── Missing values ─────────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
has_missing = missing_df['Missing Count'].sum() > 0

if has_missing:
    print('Columns with missing values:')
    print(missing_df[missing_df['Missing Count'] > 0])
    # Fill numerical with median, categorical with mode
    for col in df.columns:
        if df[col].isnull().sum() > 0:
            if df[col].dtype in ['float64', 'int64']:
                df[col].fillna(df[col].median(), inplace=True)
                print(f'  Filled [{col}] with median.')
            else:
                df[col].fillna(df[col].mode()[0], inplace=True)
                print(f'  Filled [{col}] with mode.')
else:
    print('No missing values found — dataset is complete. No imputation needed.')

In [ ]:
# ── Duplicate records ──────────────────────────────────────────────────────
dupes = df.duplicated().sum()
print(f'Duplicate rows found: {dupes}')
if dupes > 0:
    df.drop_duplicates(inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f'Duplicates removed. New shape: {df.shape}')
else:
    print('No duplicates — no action needed.')

print(f'\nFinal clean dataset shape: {df.shape}')

---
## Section 5 — Exploratory Data Analysis (EDA)

### 5.1 — Churn Distribution (Target Variable)

In [ ]:
churn_counts = df[TARGET].value_counts()
churn_pct    = df[TARGET].value_counts(normalize=True) * 100

print('Churn Distribution')
print(f'  Active  (0) : {churn_counts[0]:>5}  ({churn_pct[0]:.1f}%)')
print(f'  Churned (1) : {churn_counts[1]:>5}  ({churn_pct[1]:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(['Active (0)', 'Churned (1)'], churn_counts.values,
            color=['#27ae60', '#e74c3c'], edgecolor='black', width=0.45)
axes[0].set_title('Customer Churn Count', fontweight='bold')
axes[0].set_ylabel('Number of Customers')
axes[0].set_xlabel('Churn Status')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 25, str(v), ha='center', fontweight='bold')

axes[1].pie(churn_counts.values, labels=['Active (0)', 'Churned (1)'],
            autopct='%1.1f%%', colors=['#27ae60', '#e74c3c'],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Churn Proportion', fontweight='bold')

plt.suptitle('Netflix Customer Churn Distribution', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('01_churn_distribution.png', bbox_inches='tight')
plt.show()

### 5.2 — Churn by Gender

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.countplot(data=df, x='gender', hue=TARGET,
              palette={0: '#27ae60', 1: '#e74c3c'},
              ax=axes[0], edgecolor='black')
axes[0].set_title('Churn Count by Gender', fontweight='bold')
axes[0].set_xlabel('Gender'); axes[0].set_ylabel('Count')
axes[0].legend(title='Status', labels=['Active', 'Churned'])

churn_gender = (df.groupby('gender')[TARGET].mean() * 100)
churn_gender.plot(kind='bar', ax=axes[1], rot=0, edgecolor='black',
                  color=sns.color_palette('Set2', len(churn_gender)))
axes[1].set_title('Churn Rate (%) by Gender', fontweight='bold')
axes[1].set_xlabel('Gender'); axes[1].set_ylabel('Churn Rate (%)')
axes[1].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width()/2, p.get_height() + 0.3),
                     ha='center', fontweight='bold')

plt.suptitle('Churn Analysis by Gender', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('02_churn_gender.png', bbox_inches='tight')
plt.show()

### 5.3 — Churn by Subscription Type

In [ ]:
sub_order = ['Basic', 'Standard', 'Premium']
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.countplot(data=df, x='subscription_type', hue=TARGET, order=sub_order,
              palette={0: '#27ae60', 1: '#e74c3c'}, ax=axes[0], edgecolor='black')
axes[0].set_title('Churn Count by Subscription Type', fontweight='bold')
axes[0].set_xlabel('Subscription Type'); axes[0].set_ylabel('Count')
axes[0].legend(title='Status', labels=['Active', 'Churned'])

churn_sub = (df.groupby('subscription_type')[TARGET].mean() * 100).reindex(sub_order)
churn_sub.plot(kind='bar', ax=axes[1], rot=0, edgecolor='black',
               color=['#1abc9c', '#3498db', '#9b59b6'])
axes[1].set_title('Churn Rate (%) by Subscription Type', fontweight='bold')
axes[1].set_xlabel('Subscription Type'); axes[1].set_ylabel('Churn Rate (%)')
axes[1].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width()/2, p.get_height() + 0.3),
                     ha='center', fontweight='bold')

plt.suptitle('Churn Analysis by Subscription Type', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('03_churn_subscription.png', bbox_inches='tight')
plt.show()

### 5.4 — Age Distribution by Churn Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color, name in [(0, '#27ae60', 'Active'), (1, '#e74c3c', 'Churned')]:
    sns.kdeplot(data=df[df[TARGET] == label], x='age', ax=axes[0],
                fill=True, color=color, label=name, alpha=0.55)
axes[0].set_title('Age Distribution by Churn Status', fontweight='bold')
axes[0].set_xlabel('Age'); axes[0].set_ylabel('Density')
axes[0].legend(title='Status')

sns.boxplot(data=df, x=TARGET, y='age',
            palette={0: '#27ae60', 1: '#e74c3c'}, ax=axes[1], width=0.45)
axes[1].set_title('Age Boxplot by Churn Status', fontweight='bold')
axes[1].set_xlabel('Churned  (0=Active, 1=Churned)'); axes[1].set_ylabel('Age')

plt.suptitle('Age vs Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('04_churn_age.png', bbox_inches='tight')
plt.show()

print(f"Mean age — Active : {df[df[TARGET]==0]['age'].mean():.2f}")
print(f"Mean age — Churned: {df[df[TARGET]==1]['age'].mean():.2f}")

### 5.5 — Watch Hours & Daily Watch Time vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x=TARGET, y='watch_hours',
            palette={0: '#27ae60', 1: '#e74c3c'}, ax=axes[0], width=0.45)
axes[0].set_title('Total Watch Hours by Churn Status', fontweight='bold')
axes[0].set_xlabel('Churned  (0=Active, 1=Churned)'); axes[0].set_ylabel('Total Watch Hours')

sns.boxplot(data=df, x=TARGET, y='avg_watch_time_per_day',
            palette={0: '#27ae60', 1: '#e74c3c'}, ax=axes[1], width=0.45)
axes[1].set_title('Avg Daily Watch Time by Churn Status', fontweight='bold')
axes[1].set_xlabel('Churned  (0=Active, 1=Churned)'); axes[1].set_ylabel('Avg Watch Time / Day (hrs)')

plt.suptitle('Viewing Behaviour vs Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('05_churn_watch.png', bbox_inches='tight')
plt.show()

### 5.6 — Inactivity (Last Login Days) vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color, name in [(0, '#27ae60', 'Active'), (1, '#e74c3c', 'Churned')]:
    sns.kdeplot(data=df[df[TARGET] == label], x='last_login_days', ax=axes[0],
                fill=True, color=color, label=name, alpha=0.55)
axes[0].set_title('Days Since Last Login — Distribution', fontweight='bold')
axes[0].set_xlabel('Days Since Last Login'); axes[0].set_ylabel('Density')
axes[0].legend(title='Status')

sns.boxplot(data=df, x=TARGET, y='last_login_days',
            palette={0: '#27ae60', 1: '#e74c3c'}, ax=axes[1], width=0.45)
axes[1].set_title('Days Since Last Login — Boxplot', fontweight='bold')
axes[1].set_xlabel('Churned  (0=Active, 1=Churned)'); axes[1].set_ylabel('Days Since Last Login')

plt.suptitle('Inactivity vs Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('06_churn_last_login.png', bbox_inches='tight')
plt.show()

print(f"Mean last-login days — Active : {df[df[TARGET]==0]['last_login_days'].mean():.2f}")
print(f"Mean last-login days — Churned: {df[df[TARGET]==1]['last_login_days'].mean():.2f}")

### 5.7 — Churn by Region

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

region_order = df['region'].value_counts().index.tolist()
sns.countplot(data=df, x='region', hue=TARGET, order=region_order,
              palette={0: '#27ae60', 1: '#e74c3c'}, ax=axes[0], edgecolor='black')
axes[0].set_title('Churn Count by Region', fontweight='bold')
axes[0].set_xlabel('Region'); axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=25)
axes[0].legend(title='Status', labels=['Active', 'Churned'])

churn_region = (df.groupby('region')[TARGET].mean() * 100).sort_values(ascending=False)
churn_region.plot(kind='bar', ax=axes[1], rot=30, edgecolor='black',
                  color=sns.color_palette('Set2', len(churn_region)))
axes[1].set_title('Churn Rate (%) by Region', fontweight='bold')
axes[1].set_xlabel('Region'); axes[1].set_ylabel('Churn Rate (%)')
axes[1].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width()/2, p.get_height() + 0.3),
                     ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Churn Analysis by Region', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('07_churn_region.png', bbox_inches='tight')
plt.show()

### 5.8 — Churn by Device

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

device_order = df['device'].value_counts().index.tolist()
sns.countplot(data=df, x='device', hue=TARGET, order=device_order,
              palette={0: '#27ae60', 1: '#e74c3c'}, ax=axes[0], edgecolor='black')
axes[0].set_title('Churn Count by Device', fontweight='bold')
axes[0].set_xlabel('Device'); axes[0].set_ylabel('Count')
axes[0].legend(title='Status', labels=['Active', 'Churned'])

churn_device = (df.groupby('device')[TARGET].mean() * 100).sort_values(ascending=False)
churn_device.plot(kind='bar', ax=axes[1], rot=0, edgecolor='black',
                  color=sns.color_palette('Set2', len(churn_device)))
axes[1].set_title('Churn Rate (%) by Device', fontweight='bold')
axes[1].set_xlabel('Device'); axes[1].set_ylabel('Churn Rate (%)')
axes[1].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width()/2, p.get_height() + 0.3),
                     ha='center', fontweight='bold')

plt.suptitle('Churn Analysis by Device', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('08_churn_device.png', bbox_inches='tight')
plt.show()

### 5.9 — Churn by Payment Method

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

pm_order = df['payment_method'].value_counts().index.tolist()
sns.countplot(data=df, x='payment_method', hue=TARGET, order=pm_order,
              palette={0: '#27ae60', 1: '#e74c3c'}, ax=axes[0], edgecolor='black')
axes[0].set_title('Churn Count by Payment Method', fontweight='bold')
axes[0].set_xlabel('Payment Method'); axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=20)
axes[0].legend(title='Status', labels=['Active', 'Churned'])

churn_pm = (df.groupby('payment_method')[TARGET].mean() * 100).sort_values(ascending=False)
churn_pm.plot(kind='bar', ax=axes[1], rot=25, edgecolor='black',
              color=sns.color_palette('Set2', len(churn_pm)))
axes[1].set_title('Churn Rate (%) by Payment Method', fontweight='bold')
axes[1].set_xlabel('Payment Method'); axes[1].set_ylabel('Churn Rate (%)')
axes[1].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width()/2, p.get_height() + 0.3),
                     ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Churn Analysis by Payment Method', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('09_churn_payment.png', bbox_inches='tight')
plt.show()

### 5.10 — Churn by Favourite Genre & Number of Profiles

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

churn_genre = (df.groupby('favorite_genre')[TARGET].mean() * 100).sort_values(ascending=False)
churn_genre.plot(kind='bar', ax=axes[0], rot=0, edgecolor='black',
                 color=sns.color_palette('Set2', len(churn_genre)))
axes[0].set_title('Churn Rate (%) by Favourite Genre', fontweight='bold')
axes[0].set_xlabel('Favourite Genre'); axes[0].set_ylabel('Churn Rate (%)')
axes[0].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width()/2, p.get_height() + 0.3),
                     ha='center', fontweight='bold', fontsize=9)

churn_profiles = (df.groupby('number_of_profiles')[TARGET].mean() * 100)
churn_profiles.plot(kind='bar', ax=axes[1], rot=0, edgecolor='black',
                    color=sns.color_palette('Set2', len(churn_profiles)))
axes[1].set_title('Churn Rate (%) by Number of Profiles', fontweight='bold')
axes[1].set_xlabel('Number of Profiles'); axes[1].set_ylabel('Churn Rate (%)')
axes[1].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width()/2, p.get_height() + 0.3),
                     ha='center', fontweight='bold')

plt.suptitle('Churn by Genre and Profiles', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('10_churn_genre_profiles.png', bbox_inches='tight')
plt.show()

### 5.11 — Monthly Fee Distribution vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color, name in [(0, '#27ae60', 'Active'), (1, '#e74c3c', 'Churned')]:
    sns.kdeplot(data=df[df[TARGET] == label], x='monthly_fee', ax=axes[0],
                fill=True, color=color, label=name, alpha=0.55)
axes[0].set_title('Monthly Fee Distribution by Churn', fontweight='bold')
axes[0].set_xlabel('Monthly Fee (USD)'); axes[0].set_ylabel('Density')
axes[0].legend(title='Status')

avg_fee = df.groupby('subscription_type')['monthly_fee'].mean().reindex(['Basic','Standard','Premium'])
avg_fee.plot(kind='bar', ax=axes[1], rot=0, edgecolor='black',
             color=['#1abc9c', '#3498db', '#9b59b6'])
axes[1].set_title('Average Monthly Fee by Subscription Type', fontweight='bold')
axes[1].set_xlabel('Subscription Type'); axes[1].set_ylabel('Avg Fee (USD)')
for p in axes[1].patches:
    axes[1].annotate(f'${p.get_height():.2f}',
                     (p.get_x() + p.get_width()/2, p.get_height() + 0.05),
                     ha='center', fontweight='bold')

plt.suptitle('Monthly Fee Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('11_churn_fee.png', bbox_inches='tight')
plt.show()

### 5.12 — Correlation Heatmap

In [ ]:
num_cols = ['age', 'watch_hours', 'last_login_days', 'monthly_fee',
            'number_of_profiles', 'avg_watch_time_per_day', TARGET]
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, ax=ax, linewidths=0.5, vmin=-1, vmax=1,
            annot_kws={'size': 10})
ax.set_title('Correlation Heatmap — Numerical Features', fontweight='bold', pad=14)
plt.tight_layout()
plt.savefig('12_correlation_heatmap.png', bbox_inches='tight')
plt.show()

---
## Section 6 — Feature Engineering & ML Preparation

### 6.1 — Drop ID column and separate target

In [ ]:
# Drop customer_id (unique identifier — no predictive power)
df_ml = df.drop(columns=['customer_id']).copy()

X = df_ml.drop(columns=[TARGET])
y = df_ml[TARGET]

print('Feature matrix shape :', X.shape)
print('Target vector shape  :', y.shape)
print('Features             :', X.columns.tolist())

### 6.2 — Encode categorical features (Label Encoding)

In [ ]:
cat_features = X.select_dtypes(include='object').columns.tolist()
print('Categorical features to encode:', cat_features)

le_dict  = {}
X_encoded = X.copy()

for col in cat_features:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))
    le_dict[col] = le
    print(f'  [{col}] encoded — classes: {list(le.classes_)}')

print('\nEncoding complete. Shape:', X_encoded.shape)

### 6.3 — Scale numerical features (StandardScaler for Logistic Regression)

In [ ]:
num_features = ['age', 'watch_hours', 'last_login_days', 'monthly_fee',
                'number_of_profiles', 'avg_watch_time_per_day']

scaler   = StandardScaler()
X_scaled = X_encoded.copy()
X_scaled[num_features] = scaler.fit_transform(X_scaled[num_features])

print('StandardScaler applied to:', num_features)
print('\nSample scaled statistics:')
print(X_scaled[num_features].describe().round(3))

### 6.4 — Class imbalance check

In [ ]:
class_dist = y.value_counts(normalize=True)
ratio = class_dist.max() / class_dist.min()
print('Class distribution:')
print(class_dist)
print(f'\nImbalance ratio (majority/minority): {ratio:.2f}')
if ratio < 1.5:
    print('Status: BALANCED — class_weight="balanced" used as a soft safeguard.')
else:
    print('Status: IMBALANCED — class_weight="balanced" applied to compensate.')

### 6.5 — Train / Test Split (80 / 20, stratified)

In [ ]:
# Scaled features — for Logistic Regression
X_train_sc, X_test_sc, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

# Encoded (unscaled) features — for tree-based models
X_train_enc, X_test_enc, _, _ = train_test_split(
    X_encoded, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f'Training samples : {X_train_sc.shape[0]}  ({X_train_sc.shape[0]/len(y)*100:.0f}%)')
print(f'Testing  samples : {X_test_sc.shape[0]}  ({X_test_sc.shape[0]/len(y)*100:.0f}%)')
print(f'\nTrain churn rate : {y_train.mean()*100:.1f}%')
print(f'Test  churn rate : {y_test.mean()*100:.1f}%')

---
## Section 7 — Model Training

### 7.1 — Model 1: Logistic Regression

In [ ]:
lr = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=RANDOM_STATE
)
lr.fit(X_train_sc, y_train)

lr_pred  = lr.predict(X_test_sc)
lr_proba = lr.predict_proba(X_test_sc)[:, 1]

print('Logistic Regression — trained.')
print('\nClassification Report:')
print(classification_report(y_test, lr_pred, target_names=['Active', 'Churned']))

### 7.2 — Model 2: Decision Tree Classifier

In [ ]:
dt = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=20,
    class_weight='balanced',
    random_state=RANDOM_STATE
)
dt.fit(X_train_enc, y_train)

dt_pred  = dt.predict(X_test_enc)
dt_proba = dt.predict_proba(X_test_enc)[:, 1]

print('Decision Tree — trained.')
print('\nClassification Report:')
print(classification_report(y_test, dt_pred, target_names=['Active', 'Churned']))

### 7.3 — Model 3: Random Forest Classifier

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train_enc, y_train)

rf_pred  = rf.predict(X_test_enc)
rf_proba = rf.predict_proba(X_test_enc)[:, 1]

print('Random Forest — trained.')
print('\nClassification Report:')
print(classification_report(y_test, rf_pred, target_names=['Active', 'Churned']))

---
## Section 8 — Model Evaluation

### 8.1 — Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

model_preds = [
    ('Logistic Regression', lr_pred, axes[0]),
    ('Decision Tree',       dt_pred, axes[1]),
    ('Random Forest',       rf_pred, axes[2]),
]

for name, preds, ax in model_preds:
    cm   = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=['Active', 'Churned'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontweight='bold')

plt.suptitle('Confusion Matrices — All Three Models', fontsize=14,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('13_confusion_matrices.png', bbox_inches='tight')
plt.show()

### 8.2 — ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

roc_specs = [
    ('Logistic Regression', lr_proba, '#3498db'),
    ('Decision Tree',       dt_proba, '#e67e22'),
    ('Random Forest',       rf_proba, '#27ae60'),
]

for name, proba, color in roc_specs:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f'{name}  (AUC = {auc:.3f})', color=color, lw=2)

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Baseline (AUC = 0.500)')
ax.set_title('ROC Curves — All Models', fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.legend(loc='lower right')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig('14_roc_curves.png', bbox_inches='tight')
plt.show()

---
## Section 9 — Model Comparison

In [ ]:
def metrics(y_true, y_pred, y_proba, name):
    return {
        'Model'     : name,
        'Accuracy'  : round(accuracy_score(y_true, y_pred),  4),
        'Precision' : round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall'    : round(recall_score(y_true, y_pred),    4),
        'F1-Score'  : round(f1_score(y_true, y_pred),        4),
        'ROC-AUC'   : round(roc_auc_score(y_true, y_proba),  4),
    }

results = pd.DataFrame([
    metrics(y_test, lr_pred, lr_proba, 'Logistic Regression'),
    metrics(y_test, dt_pred, dt_proba, 'Decision Tree'),
    metrics(y_test, rf_pred, rf_proba, 'Random Forest'),
]).set_index('Model')

print('Model Comparison Table')
print('=' * 60)
print(results.to_string())
print()

results.style \
    .background_gradient(cmap='YlGn', subset=['Accuracy', 'F1-Score', 'ROC-AUC']) \
    .format('{:.4f}') \
    .set_caption('Model Performance Comparison')

In [ ]:
# Visual comparison
fig, ax = plt.subplots(figsize=(12, 5))
results.T.plot(kind='bar', ax=ax, edgecolor='black', rot=0,
               color=['#3498db', '#e67e22', '#27ae60'])
ax.set_title('Model Performance Comparison — All Metrics', fontweight='bold')
ax.set_xlabel('Metric'); ax.set_ylabel('Score')
ax.set_ylim([0, 1.12])
ax.legend(title='Model', loc='lower right')
ax.axhline(y=1.0, color='gray', linestyle='--', linewidth=0.8)
for p in ax.patches:
    if p.get_height() > 0:
        ax.annotate(f'{p.get_height():.3f}',
                    (p.get_x() + p.get_width()/2, p.get_height() + 0.005),
                    ha='center', va='bottom', fontsize=7.5, rotation=90)
plt.tight_layout()
plt.savefig('15_model_comparison.png', bbox_inches='tight')
plt.show()

---
## Section 10 — Best Model Selection

In [ ]:
best_name = results['ROC-AUC'].idxmax()
best      = results.loc[best_name]

print('=' * 55)
print(f'  BEST MODEL  :  {best_name}')
print('=' * 55)
for metric, val in best.items():
    print(f'  {metric:<12}: {val:.4f}')
print('=' * 55)
print()
print('Selection criterion: ROC-AUC is the primary metric because')
print('it evaluates discrimination across all classification thresholds,')
print('making it robust for both balanced and mildly imbalanced datasets.')

---
## Section 11 — Feature Importance (Random Forest)

In [ ]:
feat_names = X_encoded.columns.tolist()
importances = rf.feature_importances_

fi_df = pd.DataFrame({'Feature': feat_names, 'Importance': importances})
fi_df = fi_df.sort_values('Importance', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('viridis', len(fi_df))
ax.barh(fi_df['Feature'][::-1], fi_df['Importance'][::-1],
        color=colors[::-1], edgecolor='black')
ax.set_title('Random Forest — Feature Importance', fontweight='bold')
ax.set_xlabel('Importance Score (Mean Decrease in Impurity)')
ax.set_ylabel('Feature')
for i, (val, name) in enumerate(zip(fi_df['Importance'][::-1], fi_df['Feature'][::-1])):
    ax.text(val + 0.001, i, f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('16_feature_importance.png', bbox_inches='tight')
plt.show()

print('Top 5 Predictive Features:')
print(fi_df.head(5).to_string(index=False))

---
## Section 12 — Cross-Validation Robustness Check

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_lr = cross_val_score(lr, X_scaled,  y, cv=cv, scoring='roc_auc', n_jobs=-1)
cv_dt = cross_val_score(dt, X_encoded, y, cv=cv, scoring='roc_auc', n_jobs=-1)
cv_rf = cross_val_score(rf, X_encoded, y, cv=cv, scoring='roc_auc', n_jobs=-1)

cv_summary = {
    'Logistic Regression': cv_lr,
    'Decision Tree'      : cv_dt,
    'Random Forest'      : cv_rf,
}

print('5-Fold Stratified Cross-Validation  —  ROC-AUC')
print('=' * 60)
for name, scores in cv_summary.items():
    print(f'{name:<22}  Scores: {scores.round(4)}  |  '
          f'Mean: {scores.mean():.4f}  Std: {scores.std():.4f}')

fig, ax = plt.subplots(figsize=(9, 5))
bp = ax.boxplot(list(cv_summary.values()), labels=list(cv_summary.keys()),
                patch_artist=True,
                medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], ['#3498db', '#e67e22', '#27ae60']):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_title('5-Fold CV ROC-AUC — All Models', fontweight='bold')
ax.set_ylabel('ROC-AUC Score')
ax.set_xlabel('Model')
plt.tight_layout()
plt.savefig('17_cv_scores.png', bbox_inches='tight')
plt.show()

---
## Section 13 — Business Insights

In [ ]:
insights = """
PRACTICAL BUSINESS INSIGHTS — NETFLIX CHURN REDUCTION
======================================================

1. RE-ENGAGEMENT FOR INACTIVE USERS
   last_login_days is the strongest churn predictor. Customers inactive
   for 10+ days should receive personalised push notifications or emails
   highlighting new content in their favourite genre.

2. WATCH-HOUR NUDGES FOR LOW-ENGAGEMENT USERS
   Churned customers watch significantly fewer hours. Curated playlists,
   "Top Picks for You", and "Continue Watching" prompts can boost engagement.

3. SUBSCRIPTION UPGRADE INCENTIVES
   Basic-tier users are most at risk. Offering a discounted or free trial
   upgrade to Standard/Premium can reduce churn and increase revenue.

4. REGIONAL CONTENT LOCALISATION
   Regions with higher churn rates need investment in local-language content,
   regional news, sports, and culturally relevant programming.

5. PROFILE EXPANSION PROMPTS
   Single-profile households are less sticky. Prompting users to create
   family/kids profiles increases household adoption and reduces churn.

6. PAYMENT METHOD FRICTION REDUCTION
   Payment-related churn (failed renewals, expired cards) can be reduced
   by proactive billing reminders and seamless payment retry flows.

7. DEVICE-SPECIFIC APP IMPROVEMENTS
   Higher churn on specific devices signals UX/performance issues.
   Investing in app quality for those platforms (e.g., Mobile, Tablet)
   can directly reduce churn.

8. GENRE-BASED CONTENT STRATEGY
   Genres associated with higher churn indicate content gaps. Netflix
   should licence or produce more content in those genres.

9. REAL-TIME CHURN SCORING
   Deploy the Random Forest model as a production churn-risk scorer.
   Flag customers with predicted probability > 0.65 for targeted
   retention offers — discounts, exclusive previews, or loyalty rewards.

10. LOYALTY PROGRAMME
    Introduce watch-milestone rewards (e.g., "Watch 100 hrs, unlock a
    free month") or referral programmes to increase long-term retention.
"""
print(insights)

---
## Section 14 — Limitations

In [ ]:
limitations = """
PROJECT LIMITATIONS
===================

1. Dataset size (5,000 rows) may not fully represent Netflix's scale of
   200 million+ subscribers. Generalisability is limited.

2. No temporal features (subscription start date, tenure) are available,
   preventing lifecycle-based analysis (e.g., churn spikes at 3 months).

3. Label Encoding introduces artificial ordinal relationships for nominal
   categorical features (region, payment method, device). One-Hot Encoding
   would be more appropriate for linear models.

4. No hyperparameter tuning was performed. Production models should use
   GridSearchCV, RandomizedSearchCV, or Bayesian optimisation.

5. Static snapshot — the model must be retrained periodically on fresh data
   to account for concept drift and changing user behaviour.

6. External factors (competitor pricing, economic downturns, viral content
   on rival platforms) are not captured in the dataset.

7. The model predicts WHETHER a customer will churn, not WHEN or WHY.
   Survival analysis or causal models would be needed for deeper insight.
"""
print(limitations)

---
## Section 15 — Future Work

In [ ]:
future_work = """
FUTURE IMPROVEMENTS
===================

1. Hyperparameter Tuning    — GridSearchCV / Optuna for all three models.
2. Advanced Models          — XGBoost, LightGBM, or CatBoost for higher accuracy.
3. SHAP Explainability      — Per-customer churn reason explanation using SHAP values.
4. One-Hot Encoding         — Replace Label Encoding for nominal features to improve
                              Logistic Regression performance.
5. Time-Series Features     — Engineer tenure, rolling watch trends, and login streaks
                              using subscription start date if available.
6. Deep Learning            — Multi-layer perceptron (MLP) for non-linear pattern capture.
7. Production Deployment    — Wrap the best model in a REST API (Flask / FastAPI) to
                              serve real-time churn probability scores.
8. A/B Testing Integration  — Connect churn predictions to A/B tests measuring the
                              impact of retention interventions.
9. Ensemble Stacking        — Combine all three models via a meta-learner for
                              potentially higher predictive performance.
10. SMOTE / ADASYN          — If a more imbalanced real-world dataset is used, apply
                               oversampling before training.
"""
print(future_work)

---
## Section 16 — Conclusion & Key Findings

In [ ]:
print('=' * 62)
print('  CONCLUSION — Netflix Customer Churn Prediction')
print('=' * 62)

churn_rate  = df[TARGET].mean() * 100
top_region  = (df.groupby('region')[TARGET].mean()*100).idxmax()
top_sub     = (df.groupby('subscription_type')[TARGET].mean()*100).idxmax()
top_device  = (df.groupby('device')[TARGET].mean()*100).idxmax()
top3_feats  = fi_df.head(3)['Feature'].tolist()

print(f"""
This project built a complete end-to-end churn prediction pipeline:

  Dataset      : 5,000 customers × 13 usable features
  Target       : '{TARGET}'  (0=Active, 1=Churned)
  Churn Rate   : {churn_rate:.1f}%  (near-balanced dataset)

KEY FINDINGS
────────────
  * Overall churn rate        : {churn_rate:.1f}%
  * Highest churn region      : {top_region}
  * Highest churn subscription: {top_sub}
  * Highest churn device      : {top_device}
  * Top 3 predictive features : {top3_feats[0]}, {top3_feats[1]}, {top3_feats[2]}

  * Best model  : {best_name}
    Accuracy    : {results.loc[best_name,'Accuracy']:.4f}
    F1-Score    : {results.loc[best_name,'F1-Score']:.4f}
    ROC-AUC     : {results.loc[best_name,'ROC-AUC']:.4f}

SUMMARY
───────
  Inactivity (last_login_days) and low viewing engagement (watch_hours,
  avg_watch_time_per_day) are the strongest indicators of customer churn.
  The Random Forest model achieved the highest performance across all
  metrics and is recommended for production deployment as a churn-risk
  scoring engine. Ten actionable business recommendations were derived
  to help Netflix proactively reduce subscriber churn.
""")

print('=' * 62)
print('Project complete.  —  Telukuntla Yashwanth')
print('IBM SkillsBuild Data Analytics with AI | BharatCares & AICTE')
print('=' * 62)